# NISAR GCOV Colour Composite over Belgium

This notebook demonstrates how to **search, download, and visualise** NISAR L2 GCOV
(Geocoded Covariance Matrix) products over Belgium, producing false-colour composites
analogous to OPERA RTC-S1 imagery.

## NISAR GCOV product

The GCOV product is an HDF5 file containing multi-looked, geocoded covariance matrix
elements organised by frequency band:

| Band | Path in HDF5 | Physical meaning |
|------|-------------|------------------|
| L-band frequencyA | `/science/LSAR/GCOV/grids/frequencyA/HHHH` | HH backscatter power |
| L-band frequencyA | `/science/LSAR/GCOV/grids/frequencyA/HVHV` | HV backscatter power |
| S-band frequencyA | `/science/SSAR/GCOV/grids/frequencyA/VVVV` | VV backscatter power |
| S-band frequencyA | `/science/SSAR/GCOV/grids/frequencyA/VHVH` | VH backscatter power |

For the colour composite we follow the same **RTC convention**:
- **R** = co-pol amplitude  (√HHHH or √VVVV)
- **G** = cross-pol amplitude (√HVHV or √VHVH)
- **B** = co-pol / cross-pol ratio

## Archive access

NISAR GCOV products are available from two ASF collections searched automatically:
- **Public Beta** (`NISAR`, processingLevel=`GCOV`) — open access
- **Early Adopter L2** (`C4052499921-ASF`) — gated; requires EA credentials in `~/.netrc`

## Workflow

1. Search for GCOV products over Belgium
2. Inspect the first product (metadata + file structure)
3. Download the HDF5 file
4. Extract L-band covariance layers
5. Build and display a false-colour composite
6. Compare colour-stretch options (P2–P98 vs. fixed ranges)

In [ ]:
%matplotlib inline

import warnings
warnings.filterwarnings('ignore')

import os
import numpy as np
import matplotlib.pyplot as plt
import h5py
import requests
import netrc

import asf_search

from rs_tools.config import BoundingBox, SearchConfig
from rs_tools.search import search_archive
from rs_tools.datasets.catalog import get as get_dataset
from rs_tools.archives.nasa import _NISAR_EA_L2

## 1. Define AOI & Parameters

Belgium bounding box and date range. We start with a single GCOV product to
explore the file structure before scaling up.

In [ ]:
# Belgium-wide bounding box
bbox_belgium = BoundingBox(west=2.54, south=49.50, east=6.41, north=51.50)

# Date range — adjust to a period when NISAR data is available over Belgium
START_DATE = '2025-01-01'
END_DATE   = '2026-05-28'

# Output directory for downloaded HDF5 files
WORKDIR = os.path.expanduser('~/RS_applications/Applications/GCOV/BelgiumColors')
os.makedirs(WORKDIR, exist_ok=True)

print(f'AOI:      {bbox_belgium}')
print(f'Period:   {START_DATE} → {END_DATE}')
print(f'WORKDIR:  {WORKDIR}')

# Confirm catalog entry
gcov_ds = get_dataset('NISAR_L2_GCOV')
print(f'\nDataset: {gcov_ds.name}')
print(f'  resolution : {gcov_ds.spatial_resolution}')
print(f'  tags       : {gcov_ds.tags}')

## 2. Search for GCOV Products

We use the `rs_tools` `NASAArchive` which automatically searches both the
public Beta collection and the Early Adopter L2 collection (`C4052499921-ASF`).
EA results are silently skipped if you do not have EA credentials.

In [ ]:
config = SearchConfig(
    start_date=START_DATE,
    end_date=END_DATE,
    bbox=bbox_belgium,
    collections=['NISAR_L2_GCOV'],
    limit=50,
    include_ea=True,   # also query EA L2 collection
)

items = search_archive('nasa', config)
print(f'Found {len(items)} GCOV granule(s).')

for it in items[:10]:
    props = it.get('properties', {})
    assets = it.get('assets', {})
    asset_keys = list(assets.keys())
    print(f"  {it['id']}")
    print(f"    datetime : {props.get('datetime', 'n/a')}")
    print(f"    platform : {props.get('platform', 'n/a')}")
    print(f"    assets   : {asset_keys}")

## 3. Inspect First Product

Pick the first available granule and examine its download URL and metadata.

In [ ]:
if not items:
    raise RuntimeError(
        'No GCOV products found. '
        'Check date range, AOI, and EA credentials in ~/.netrc.'
    )

item = items[0]
print('Selected granule:', item['id'])
print('Properties:')
for k, v in item.get('properties', {}).items():
    print(f'  {k:25s}: {v}')

print('\nAssets:')
for key, asset in item.get('assets', {}).items():
    print(f'  {key:15s}: {asset.get("href", "")}')

# Resolve download URL — the main .h5 file
assets = item.get('assets', {})
# GCOV: single HDF5 — stored under key 'data' (fallback to first key)
h5_asset = assets.get('data') or next(iter(assets.values()), None)
h5_url = h5_asset['href'] if h5_asset else None
print(f'\nDownload URL: {h5_url}')

## 4. Download HDF5 File

Download the GCOV HDF5 file to `WORKDIR`. Subsequent runs skip re-downloading
if the file already exists on disk.

> **Auth**: NASA Earthdata credentials must be in `~/.netrc` under
> `machine urs.earthdata.nasa.gov`.

In [ ]:
def _get_auth():
    """Return (username, password) from ~/.netrc for urs.earthdata.nasa.gov."""
    try:
        nrc = netrc.netrc()
        auth = nrc.authenticators('urs.earthdata.nasa.gov')
        if auth:
            return auth[0], auth[2]
    except FileNotFoundError:
        pass
    return None, None


def download_file(url, dest_dir, chunk_size=1024 * 1024):
    """Download *url* to *dest_dir*, returning the local path.
    
    Skips download if the file already exists.
    Follows NASA Earthdata OAuth redirects via ~/.netrc credentials.
    """
    fname = os.path.basename(url.split('?')[0])  # strip query params
    local = os.path.join(dest_dir, fname)
    if os.path.exists(local):
        print(f'Already on disk: {local}')
        return local

    user, pw = _get_auth()
    session = requests.Session()
    if user:
        session.auth = (user, pw)

    print(f'Downloading {fname} …', end=' ', flush=True)
    resp = session.get(url, stream=True, timeout=120)
    resp.raise_for_status()
    size = 0
    with open(local, 'wb') as fh:
        for chunk in resp.iter_content(chunk_size=chunk_size):
            fh.write(chunk)
            size += len(chunk)
    print(f'{size / 1e6:.1f} MB → {local}')
    return local


h5_path = download_file(h5_url, WORKDIR)

## 5. Explore HDF5 Structure

Print the top-level group structure of the GCOV file so we can locate the
covariance layers and geolocation information.

In [ ]:
def print_h5_tree(path, max_depth=4):
    """Recursively print the HDF5 group/dataset hierarchy."""
    def _visit(name, obj):
        depth = name.count('/')
        if depth > max_depth:
            return
        indent = '  ' * depth
        if isinstance(obj, h5py.Dataset):
            print(f'{indent}[DS] {name}  shape={obj.shape}  dtype={obj.dtype}')
        else:
            print(f'{indent}[G]  {name}/')

    with h5py.File(path, 'r') as f:
        print(f'Root groups: {list(f.keys())}\n')
        f.visititems(_visit)

print_h5_tree(h5_path)

## 6. Load Covariance Layers

Read the diagonal covariance terms for L-band frequencyA (HHHH, HVHV) or
S-band (VVVV, VHVH) depending on which band is present in the file.

The diagonal terms represent the backscatter power in each polarisation channel,
directly analogous to the VV / VH GeoTIFF layers in an OPERA RTC product.

In [ ]:
# GCOV HDF5 paths for the diagonal covariance terms
_LBAND_COPOL   = '/science/LSAR/GCOV/grids/frequencyA/HHHH'
_LBAND_XPOL    = '/science/LSAR/GCOV/grids/frequencyA/HVHV'
_SBAND_COPOL   = '/science/SSAR/GCOV/grids/frequencyA/VVVV'
_SBAND_XPOL    = '/science/SSAR/GCOV/grids/frequencyA/VHVH'

# Geolocation paths (same structure for LSAR/SSAR)
_LBAND_X       = '/science/LSAR/GCOV/grids/frequencyA/xCoordinates'
_LBAND_Y       = '/science/LSAR/GCOV/grids/frequencyA/yCoordinates'
_LBAND_PROJ    = '/science/LSAR/GCOV/grids/frequencyA/projection'


def load_gcov_channels(h5_path, subsample=4):
    """Load the best available co-pol + cross-pol power layers.

    Prefers L-band (HH/HV); falls back to S-band (VV/VH).
    subsample: read every N-th row/col (default 4 → ~80 m for 20 m native),
               reducing memory by ~16× while retaining full visual quality.
    Returns (co_pol, x_pol, x_coords, y_coords, proj_wkt, band_label).
    All power arrays are float32 with NaN for missing data.
    """
    s = slice(None, None, subsample)
    with h5py.File(h5_path, 'r') as f:
        # Try L-band first
        if _LBAND_COPOL in f and _LBAND_XPOL in f:
            co  = f[_LBAND_COPOL][s, s].astype(np.float32)
            xp  = f[_LBAND_XPOL][s, s].astype(np.float32)
            xc  = f[_LBAND_X][s] if _LBAND_X in f else None
            yc  = f[_LBAND_Y][s] if _LBAND_Y in f else None
            prj = f[_LBAND_PROJ][()] if _LBAND_PROJ in f else None
            band_label = 'L-band (HH/HV)'
        elif _SBAND_COPOL in f and _SBAND_XPOL in f:
            sband_x = '/science/SSAR/GCOV/grids/frequencyA/xCoordinates'
            sband_y = '/science/SSAR/GCOV/grids/frequencyA/yCoordinates'
            sband_p = '/science/SSAR/GCOV/grids/frequencyA/projection'
            co  = f[_SBAND_COPOL][s, s].astype(np.float32)
            xp  = f[_SBAND_XPOL][s, s].astype(np.float32)
            xc  = f[sband_x][s] if sband_x in f else None
            yc  = f[sband_y][s] if sband_y in f else None
            prj = f[sband_p][()] if sband_p in f else None
            band_label = 'S-band (VV/VH)'
        else:
            raise ValueError(
                'Neither L-band HHHH/HVHV nor S-band VVVV/VHVH found. '
                f'Run print_h5_tree("{h5_path}") to inspect the structure.'
            )

    # Replace fill / invalid values with NaN
    co[co < 0] = np.nan
    xp[xp < 0] = np.nan

    if isinstance(prj, bytes):
        prj = prj.decode()

    eff = f'~{20 * subsample} m effective' if subsample > 1 else 'native 20 m'
    print(f'Loaded {band_label}  (subsample={subsample}, {eff})')
    print(f'  co-pol  shape={co.shape}  valid={np.sum(np.isfinite(co)):,}')
    print(f'  cross-pol shape={xp.shape}  valid={np.sum(np.isfinite(xp)):,}')
    return co, xp, xc, yc, prj, band_label


co_pol, x_pol, x_coords, y_coords, proj_wkt, band_label = load_gcov_channels(h5_path)

## 7. Amplitude Statistics

Compute percentile statistics for the co-pol and cross-pol channels.
We work in **amplitude** (√power) to match the RTC convention and make
histograms visually comparable.

In [ ]:
rng = np.random.default_rng(42)
MAX_PIXELS = 5_000_000  # subsample to keep memory manageable

# Convert power → amplitude; mask invalid pixels
co_amp = np.sqrt(co_pol[np.isfinite(co_pol) & (co_pol > 0)])
xp_amp = np.sqrt(x_pol[np.isfinite(x_pol)  & (x_pol  > 0)])

if len(co_amp) > MAX_PIXELS:
    co_amp = co_amp[rng.choice(len(co_amp), MAX_PIXELS, replace=False)]
if len(xp_amp) > MAX_PIXELS:
    xp_amp = xp_amp[rng.choice(len(xp_amp), MAX_PIXELS, replace=False)]

percentiles = [1, 2, 5, 25, 50, 75, 95, 98, 99]

print(f'Amplitude statistics — {band_label}')
print(f'{"":-<60}')
print(f'{"Pct":>5}  {"Co-pol":>12}  {"Cross-pol":>12}')
for p in percentiles:
    print(f'  P{p:<3d}  {np.percentile(co_amp, p):>12.5f}  {np.percentile(xp_amp, p):>12.5f}')

# Suggested colour ranges (P2–P98, same logic as RTC notebook)
co_range = (float(np.percentile(co_amp, 2)), float(np.percentile(co_amp, 98)))
xp_range = (float(np.percentile(xp_amp, 2)), float(np.percentile(xp_amp, 98)))
print(f'\nSuggested colour ranges (P2–P98):')
print(f'  co-pol  : {co_range}')
print(f'  cross-pol: {xp_range}')

## 8. Amplitude Histograms

Visualise the amplitude distributions with the P2–P98 colour ranges overlaid.

In [ ]:
pol_labels = band_label.split('(')[1].rstrip(')').split('/')  # ['HH','HV'] or ['VV','VH']
co_lbl  = pol_labels[0].strip()
xp_lbl  = pol_labels[1].strip()

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

for ax, amp, rng, pol, color in [
    (axes[0], co_amp, co_range, co_lbl, 'steelblue'),
    (axes[1], xp_amp, xp_range, xp_lbl, 'darkorange'),
]:
    ax.hist(amp, bins=300, density=True, alpha=0.7, color=color,
            label=f'{pol} amplitude')
    ax.axvline(rng[0], color='lime', ls='--', lw=1.5, label=f'P2  = {rng[0]:.4f}')
    ax.axvline(rng[1], color='lime', ls='-',  lw=1.5, label=f'P98 = {rng[1]:.4f}')
    ax.set_title(f'{pol} amplitude — {band_label}', fontsize=11)
    ax.set_xlabel('Amplitude (√power)')
    ax.set_ylabel('Density')
    ax.legend(fontsize=8)

plt.suptitle(f'NISAR GCOV amplitude distributions  —  {item["id"]}', fontsize=12)
plt.tight_layout()
plt.savefig(os.path.join(WORKDIR, 'gcov_amplitude_hist.png'), dpi=150, bbox_inches='tight')
plt.show()

## 9. False-Colour Composite

Build an RGB image following the standard SAR colour-composite convention:

| Channel | Source      | Meaning                        |
|---------|-------------|--------------------------------|
| **R**   | co-pol amp  | Surface/volume scattering      |
| **G**   | cross-pol amp | Volume / vegetation / crops  |
| **B**   | co/cross ratio | Double-bounce / urban        |

Each channel is linearly stretched to [0, 1] using the P2–P98 ranges above.

In [ ]:
def gcov_composite(co_power, xp_power,
                   co_range=(0.05, 0.55),
                   xp_range=(0.02, 0.25)):
    """Create a false-colour RGB composite from GCOV covariance power layers.

    Parameters
    ----------
    co_power : ndarray
        Co-polarisation power (HHHH or VVVV), float32.
    xp_power : ndarray
        Cross-polarisation power (HVHV or VHVH), float32.
    co_range : (float, float)
        Amplitude (√power) clipping range for the co-pol channel.
    xp_range : (float, float)
        Amplitude (√power) clipping range for the cross-pol channel.

    Returns
    -------
    rgb : ndarray, shape (H, W, 3), float32 in [0, 1]
        False-colour composite: R=co-pol, G=cross-pol, B=co/cross ratio.
    """
    def _stretch(arr, lo, hi):
        out = np.clip((arr - lo) / (hi - lo + 1e-12), 0.0, 1.0)
        out[~np.isfinite(arr)] = 0.0
        return out.astype(np.float32)

    # Amplitude = sqrt(power)
    co_amp = np.sqrt(np.clip(co_power, 0, None))
    xp_amp = np.sqrt(np.clip(xp_power, 0, None))

    R = _stretch(co_amp, co_range[0], co_range[1])
    G = _stretch(xp_amp, xp_range[0], xp_range[1])

    # Blue = co/cross ratio, stretched to [0, 1]
    with np.errstate(invalid='ignore', divide='ignore'):
        ratio = co_amp / (xp_amp + 1e-12)
    ratio_lo, ratio_hi = 0.5, 5.0  # typical range for SAR co/cross ratio
    B = _stretch(ratio, ratio_lo, ratio_hi)

    return np.stack([R, G, B], axis=-1)


rgb = gcov_composite(co_pol, x_pol, co_range=co_range, xp_range=xp_range)

fig, ax = plt.subplots(figsize=(12, 10))
ax.imshow(rgb, origin='upper')
ax.set_axis_off()
ax.set_title(
    f'NISAR GCOV false-colour composite — {band_label}\n'
    f'{item["id"]}\n'
    f'R={co_lbl} amp  G={xp_lbl} amp  B={co_lbl}/{xp_lbl} ratio',
    fontsize=10,
)
plt.tight_layout()
plt.savefig(os.path.join(WORKDIR, 'gcov_composite.png'), dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved → {WORKDIR}/gcov_composite.png')

## 10. Compare Colour-Stretch Options

Show three side-by-side composites with different amplitude stretch ranges
to help pick the best visualisation for Belgium land cover.

In [ ]:
# Three stretch options to compare
stretch_options = [
    ('P2–P98 (auto)',   co_range,                           xp_range),
    ('Tight (P5–P95)',  (float(np.percentile(co_amp, 5)),   float(np.percentile(co_amp, 95))),
                        (float(np.percentile(xp_amp, 5)),   float(np.percentile(xp_amp, 95)))),
    ('Wide  (P1–P99)',  (float(np.percentile(co_amp, 1)),   float(np.percentile(co_amp, 99))),
                        (float(np.percentile(xp_amp, 1)),   float(np.percentile(xp_amp, 99)))),
]

fig, axes = plt.subplots(1, 3, figsize=(18, 7))

for ax, (label, cr, xr) in zip(axes, stretch_options):
    rgb_i = gcov_composite(co_pol, x_pol, co_range=cr, xp_range=xr)
    ax.imshow(rgb_i, origin='upper')
    ax.set_axis_off()
    ax.set_title(
        f'{label}\n'
        f'{co_lbl}: [{cr[0]:.3f}, {cr[1]:.3f}]\n'
        f'{xp_lbl}: [{xr[0]:.3f}, {xr[1]:.3f}]',
        fontsize=9,
    )

plt.suptitle(f'NISAR GCOV — colour stretch comparison  |  {band_label}', fontsize=12)
plt.tight_layout()
plt.savefig(os.path.join(WORKDIR, 'gcov_stretch_comparison.png'), dpi=150, bbox_inches='tight')
plt.show()

## 11. Summary

Print the recommended colour ranges ready for use in further analysis.

In [ ]:
print('=' * 60)
print('NISAR GCOV  —  Recommended Colour Ranges for Belgium')
print('=' * 60)
print(f'  Band: {band_label}')
print(f'  Co-pol  ({co_lbl}):   amplitude range = {co_range}')
print(f'  Cross-pol ({xp_lbl}): amplitude range = {xp_range}')
print()
print('Usage:')
print('  rgb = gcov_composite(')
print('            co_power, x_power,')
print(f'            co_range={co_range},')
print(f'            xp_range={xp_range})')
print()
print(f'Output files in: {WORKDIR}')